In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

import pandas as pd
from sentence_transformers import SentenceTransformer

import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.preprocessing import OneHotEncoder

import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader

import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

#定义并立即执行固定种子的函数
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42) 
print("随机种子已固定为 42")

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2') 

news_df = pd.read_csv('news.tsv', sep='\t', header=None,
                      names=['news_id', 'category', 'sub_category', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities'])

# 拼接标题和摘要
news_df['full_text'] = news_df['title'].fillna('') + " " + news_df['abstract'].fillna('')
print("正在利用大语言模型提取新闻的深层语义 Embedding...")

sentences = news_df['full_text'].head(1000).tolist()
embeddings = model.encode(sentences, show_progress_bar=True)

print(f"新闻已被转化为稠密向量空间，矩阵形状为: {embeddings.shape}")

In [ ]:
print("开始全量提取训练集所有新闻的 Embedding（大约几万条，请耐心等待1-2分钟）...")
all_sentences = news_df['full_text'].tolist()
all_embeddings = model.encode(all_sentences, batch_size=64, show_progress_bar=True)

# 建立一个字典，方便通过新闻 ID 瞬间查到它的向量 (类似于 Hash 映射)
news_id_list = news_df['news_id'].tolist()
news_embedding_dict = {news_id: vec for news_id, vec in zip(news_id_list, all_embeddings)}
print("全量新闻语义空间构建完毕！")

#读取用户行为日志表 behaviors.tsv
behaviors_df = pd.read_csv('behaviors.tsv', sep='\t', header=None,
                             names=['impression_id', 'user_id', 'time', 'history', 'impressions'])

#构建训练样本特征集
X_list = []
y_list = []

print("正在将大模型语义向量融合进用户点击行为流中...")
# 遍历前 5000 条用户点击日志（先用5000条做局部实验，防止内存溢出）
for _, row in behaviors_df.head(5000).iterrows():
    history_str = row['history']
    impressions_str = row['impressions']
    
    #过滤掉历史记录为空的边缘脏数据
    if pd.isna(history_str) or pd.isna(impressions_str):
        continue
        
    #计算该用户的兴趣向量（将历史看过的新闻向量求平均）
    history_news_ids = history_str.split()
    history_vecs = [news_embedding_dict[nid] for nid in history_news_ids if nid in news_embedding_dict]
    
    if len(history_vecs) == 0:
        continue
    user_vector = np.mean(history_vecs, axis=0)
    
    #拆解系统展现给他的新闻列表，提取正负样本标签
    imprs = impressions_str.split()
    for impr in imprs:
        target_news_id, label = impr.split('-') 
        label = int(label)
        
        # 如果展现的这部新闻在大模型词表里
        if target_news_id in news_embedding_dict:
            target_news_vector = news_embedding_dict[target_news_id] # 目标新闻向量
            
            # 【核心特征融合】：把用户兴趣向量、目标新闻向量、以及它们的内积（相似度）横向拼在一起
            similarity = np.dot(user_vector, target_news_vector)
            
            # 组合成当前样本的复合大模型语义特征
            feature_vector = np.hstack([user_vector, target_news_vector, [similarity]])
            
            X_list.append(feature_vector)
            y_list.append(label)

X_fine_tune = np.array(X_list)
y_fine_tune = np.array(y_list)

print(f"最终喂给模型的特征矩阵形状为: {X_fine_tune.shape}")
print(f"标签矩阵形状为: {y_fine_tune.shape} (正负样本比: {np.sum(y_fine_tune)} / {len(y_fine_tune) - np.sum(y_fine_tune)})")

In [ ]:
#严格划分训练集与测试集
print("正在划分数据集...")
X_train, X_test, y_train, y_test = train_test_split(
    X_fine_tune, y_fine_tune, test_size=0.2, random_state=42, stratify=y_fine_tune
)

print("正在训练 LightGBM 树算子（设定50棵树，每棵树32个叶子节点）...")
lgb_model = lgb.LGBMClassifier(
    n_estimators=50, 
    num_leaves=32, 
    random_state=42, 
    n_jobs=-1,
    learning_rate=0.1
)
lgb_model.fit(X_train, y_train)

#利用树的叶子节点索引进行 One-Hot 变换
print("正在将连续的向量特征通过树模型转换为离散的空间基底...")
train_leaf = lgb_model.predict(X_train, pred_leaf=True)
test_leaf = lgb_model.predict(X_test, pred_leaf=True)

#初始化独热编码器，把叶子 ID 组合变成工业高维稀疏特征
leaf_encoder = OneHotEncoder(categories='auto')
X_train_lr = leaf_encoder.fit_transform(train_leaf)
X_test_lr = leaf_encoder.transform(test_leaf)

print(f"坐标变换完毕！下游 LR 面对的解空间维度为: {X_train_lr.shape}")

#逻辑回归 (LR) 接收稀疏大盘进行光滑预测
print("正在训练带有 L2 正则化约束的 Logistic Regression 分类器...")
#引入强大的 L2 正则化（C=0.1 即惩罚力度较大），压制高维稀疏矩阵的过拟合风险
lr_model = LogisticRegression(C=0.1, max_iter=1000, solver='lbfgs', n_jobs=-1)
lr_model.fit(X_train_lr, y_train)

#预测点击概率
y_pred_prob = lr_model.predict_proba(X_test_lr)[:, 1]

#计算推荐系统最核心的两个评估指标
test_auc = roc_auc_score(y_test, y_pred_prob)
test_logloss = log_loss(y_test, y_pred_prob)

print(f"测试集 AUC 评分  : {test_auc:.4f}")
print(f"交叉熵损失 LogLoss: {test_logloss:.4f}")

In [ ]:
#提取 LightGBM 算子中 769 维特征的重要性
importance = lgb_model.feature_importances_

#还原特征的名字
# 0~383 是用户兴趣向量，384~767 是新闻向量，768 是内积相似度
feature_names = [f"user_emb_{i}" for i in range(384)] + \
                [f"news_emb_{i}" for i in range(384)] + \
                ["cosine_similarity"]

#组装成 DataFrame 并按重要性降序排列
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)

#打印前 15 个最核心的物理分裂特征
print(importance_df.head(15).to_string(index=False))
sim_rank = importance_df[importance_df['Feature'] == 'cosine_similarity']
print("\n" + "-"*40)
print(f"数学内积特征 [cosine_similarity] 的表现：\n{sim_rank.to_string(index=False)}")
print("="*40)

In [ ]:
#读取并加工验证集
print("正在读取验证集新闻并利用大模型转换为 Embedding...")
dev_news_df = pd.read_csv('dev_news.tsv', sep='\t', header=None,
                          names=['news_id', 'category', 'sub_category', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities'])

dev_news_df['full_text'] = dev_news_df['title'].fillna('') + " " + dev_news_df['abstract'].fillna('')

#使用同一个大模型 model 编码验证集的所有新闻
dev_sentences = dev_news_df['full_text'].tolist()
dev_embeddings = model.encode(dev_sentences, batch_size=64, show_progress_bar=True)

#建立验证集的新闻字典
dev_news_id_list = dev_news_df['news_id'].tolist()
dev_news_embedding_dict = {nid: vec for nid, vec in zip(dev_news_id_list, dev_embeddings)}

#读取验证集的行为日志
print("\n正在读取验证集用户行为日志...")
dev_behaviors_df = pd.read_csv('dev_behaviors.tsv', sep='\t', header=None,
                               names=['impression_id', 'user_id', 'time', 'history', 'impressions'])


#构建验证集的 769 维特征矩阵
X_dev_list = []
y_dev_list = []

print("正在构建验证集的复合特征大盘（局部抽取 5000 条进行实战检验）...")
for _, row in dev_behaviors_df.head(5000).iterrows():
    history_str = row['history']
    impressions_str = row['impressions']
    
    if pd.isna(history_str) or pd.isna(impressions_str):
        continue
        
    #计算用户兴趣质心 (注意：如果验证集里的新闻历史在训练集字典里，优先去训练集字典查，互补)
    history_news_ids = history_str.split()
    history_vecs = []
    for nid in history_news_ids:
        if nid in dev_news_embedding_dict:
            history_vecs.append(dev_news_embedding_dict[nid])
        elif nid in news_embedding_dict: 
            history_vecs.append(news_embedding_dict[nid])
            
    if len(history_vecs) == 0:
        continue
    user_vector = np.mean(history_vecs, axis=0)
    
    #拆解展示列表
    imprs = impressions_str.split()
    for impr in imprs:
        target_news_id, label = impr.split('-')
        label = int(label)
        
        #提取目标新闻向量
        target_news_vector = None
        if target_news_id in dev_news_embedding_dict:
            target_news_vector = dev_news_embedding_dict[target_news_id]
        elif target_news_id in news_embedding_dict:
            target_news_vector = news_embedding_dict[target_news_id]
            
        if target_news_vector is not None:
            similarity = np.dot(user_vector, target_news_vector)
            feature_vector = np.hstack([user_vector, target_news_vector, [similarity]])
            X_dev_list.append(feature_vector)
            y_dev_list.append(label)

X_dev = np.array(X_dev_list)
y_dev = np.array(y_dev_list)

#让 LightGBM 把验证集切分成叶子节点
dev_leaf = lgb_model.predict(X_dev, pred_leaf=True)
#用训练集训练好的 OneHotEncoder 进行完全相同的基底坐标变换
X_dev_lr = leaf_encoder.transform(dev_leaf)
#用训练好的 LR 模型预测点击概率
y_dev_pred_prob = lr_model.predict_proba(X_dev_lr)[:, 1]

#计算真正的线上泛化分数
final_dev_auc = roc_auc_score(y_dev, y_dev_pred_prob)
final_dev_logloss = log_loss(y_dev, y_dev_pred_prob)

print(f"验证集（Dev）终极泛化 AUC : {final_dev_auc:.4f}")
print(f"验证集（Dev）终极 LogLoss: {final_dev_logloss:.4f}")

In [ ]:
# 设定统一的序列几何维度
MAX_SEQ_LEN = 20  # 每个人保留最近看过的20篇新闻
EMBED_DIM = 384   # 大模型吐出的向量维度

#加载训练日志
train_behaviors_df = pd.read_csv('behaviors.tsv', sep='\t', header=None,
                                 names=['impression_id', 'user_id', 'time', 'history', 'impressions'])

X_target_list = []   # 存储目标候选新闻向量
X_history_list = []  # 存储用户历史新闻序列矩阵
y_list = []          # 存储0/1标签

#遍历行为日志构建深度流
for _, row in train_behaviors_df.head(20000).iterrows(): # 先抽取20000条行为进行深度闭环
    history_str = row['history']
    impressions_str = row['impressions']
    
    if pd.isna(history_str) or pd.isna(impressions_str):
        continue
        
    #解析历史观看序列
    history_news_ids = history_str.split()
    history_vecs = []
    for nid in history_news_ids:
        if nid in news_embedding_dict: # 这里的 news_embedding_dict 是之前离线做好的大模型字典
            history_vecs.append(news_embedding_dict[nid])
            
    if len(history_vecs) == 0:
        continue
        
   
    if len(history_vecs) >= MAX_SEQ_LEN:
        #如果历史太长，切片截取最近的 MAX_SEQ_LEN 篇
        history_vecs = history_vecs[-MAX_SEQ_LEN:]
    else:
        #如果不够长，在前面用 384维全零向量补齐
        pad_len = MAX_SEQ_LEN - len(history_vecs)
        pad_vecs = [np.zeros(EMBED_DIM) for _ in range(pad_len)]
        history_vecs = pad_vecs + history_vecs
        
    #将其转化为 (MAX_SEQ_LEN, EMBED_DIM) = (10, 384) 的标准行为矩阵
    history_matrix = np.array(history_vecs)
    
    #拆解当前展现列表的曝光样本
    imprs = impressions_str.split()
    for impr in imprs:
        target_news_id, label = impr.split('-')
        label = int(label)
        
        if target_news_id in news_embedding_dict:
            target_vector = news_embedding_dict[target_news_id]
            
            X_target_list.append(target_vector)
            X_history_list.append(history_matrix)
            y_list.append(label)

print(f"特征重构完毕(采用流式 List 存储，已免疫 MemoryError)")
print(f"目标候选新闻总样本数: {len(X_target_list)}，单条维度: [{EMBED_DIM}]")
print(f"用户历史序列总样本数: {len(X_history_list)}，单条维度: [{MAX_SEQ_LEN}, {EMBED_DIM}]")
print(f"标签总样本数: {len(y_list)}")

#封装成标准的 PyTorch 工业级流式 DataLoader
# 先定义标准流式数据集类
class RecommendationDataset(Dataset):
    def __init__(self, target, history, labels):
        self.target = target
        self.history = history
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        # 核心保命优化：每次迭代只把当前这一条小数据动态转成 PyTorch Tensor，开销几乎为 0
        return (torch.tensor(self.target[idx], dtype=torch.float32), 
                torch.tensor(self.history[idx], dtype=torch.float32), 
                torch.tensor(self.labels[idx], dtype=torch.float32))

#再实例化（直接传入原始 Python List）
train_dataset = RecommendationDataset(X_target_list, X_history_list, y_list)

#开启 Batch 迭代流水线，每批喂入 256 条样本
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

In [ ]:
#定义 DIN 的核心组件与全链路网络
class ActivationUnit(nn.Module):
    """
    DIN 的注意力激活单元：计算当前候选新闻与历史看过的每一篇新闻之间的语义交叉强弱
    """
    def __init__(self, embed_dim):
        super(ActivationUnit, self).__init__()
        #显式交叉特征总维度：4 * embed_dim
        self.fc = nn.Sequential(
            nn.Linear(embed_dim * 4, 64),
            nn.PReLU(),
            nn.Linear(64, 1) #吐出注意力权重分
        )

    def forward(self, query, facts):
        B, T, D = facts.shape
        query = query.unsqueeze(1).expand(-1, T, -1) 
        #数学交叉显式特征：[候选, 历史, 候选-历史, 候选*历史]
        cross_features = torch.cat([query, facts, query - facts, query * facts], dim=-1)
        attention_scores = self.fc(cross_features) 
        return attention_scores.squeeze(-1)


class DeepInterestNetwork(nn.Module):
    def __init__(self, embed_dim=384):
        super(DeepInterestNetwork, self).__init__()
        self.attention_unit = ActivationUnit(embed_dim)
        
        #精排层 MLP
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, 128),
            nn.PReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.PReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid() # 端到端预估出绝对点击概率，对齐 LogLoss
        )

    def forward(self, target_news_emb, history_news_embs):
        #计算注意力局部激活权重
        attention_weights = self.attention_unit(target_news_emb, history_news_embs)
        attention_weights = F.softmax(attention_weights, dim=-1).unsqueeze(-1) 
        
        #矩阵乘法：打破均值，完成用户长期多峰偏好的动态唤醒权重池化
        user_dynamic_interest = torch.bmm(attention_weights.transpose(1, 2), history_news_embs).squeeze(1)
        
        #动态兴趣与目标新闻无损级联拼接
        inp = torch.cat([user_dynamic_interest, target_news_emb], dim=-1)
        output_prob = self.mlp(inp).squeeze(-1)
        return output_prob

#实例化模型、损失函数与优化器
#检测是否有 GPU 加速，有的话直接上 CUDA，速度飞起
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前计算后端运行在: {device}")

din_model = DeepInterestNetwork(embed_dim=384).to(device)
criterion = nn.BCELoss() #二分类交叉熵损失，直接用来死磕和压低 LogLoss
optimizer = torch.optim.Adam(din_model.parameters(), lr=0.001, weight_decay=1e-5)

#开启端到端的深度网络迭代训练循环
epochs = 3 # 设定迭代 3 轮
for epoch in range(epochs):
    din_model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch_target, batch_history, batch_labels in train_loader:
        # 数据移交至计算核心 (CPU 或 GPU)
        batch_target = batch_target.to(device)
        batch_history = batch_history.to(device)
        batch_labels = batch_labels.to(device)
        
        # a. 梯度清零
        optimizer.zero_grad()
        
        # b. 前向传播：网络全自动完成动态 Target-Attention 算分
        predictions = din_model(batch_target, batch_history)
        
        # c. 计算 LogLoss 损失
        loss = criterion(predictions, batch_labels)
        
        # d. 反向传播：误差梯度一路无损回传，修正每一层交叉参数
        loss.backward()
        
        # e. 优化器更新参数
        optimizer.step()
        
        total_loss += loss.item() * batch_target.size(0)
        all_preds.extend(predictions.detach().cpu().numpy())
        all_labels.extend(batch_labels.cpu().numpy())
        
    # 计算当前 Epoch 的宏观平均评价指标
    epoch_loss = total_loss /len(train_dataset)
    epoch_auc = roc_auc_score(all_labels, all_preds)
    print(f"🏁 Epoch [{epoch+1}/{epochs}] 结束 -> 训练集平均 LogLoss: {epoch_loss:.4f} | 排序指标 AUC: {epoch_auc:.4f}")

In [ ]:
#统一几何维度参数（必须和训练时完全一致）
MAX_SEQ_LEN = 20  
EMBED_DIM = 384   

#读取验证集行为日志
dev_behaviors_df = pd.read_csv('dev_behaviors.tsv', sep='\t', header=None,
                               names=['impression_id', 'user_id', 'time', 'history', 'impressions'])

X_dev_target_list = []   # 存储验证集目标候选新闻向量
X_dev_history_list = []  # 存储验证集用户历史新闻序列矩阵
y_dev_list = []          # 存储验证集真实 0/1 标签

#开始抽样融合
for _, row in dev_behaviors_df.head(5000).iterrows(): # 【保命调优1：缩减大盘到 5000 条】
    history_str = row['history']
    impressions_str = row['impressions']
    
    if pd.isna(history_str) or pd.isna(impressions_str):
        continue
        
    #解析历史
    history_news_ids = history_str.split()
    history_vecs = []
    for nid in history_news_ids:
        if nid in news_embedding_dict: 
            history_vecs.append(news_embedding_dict[nid])
            
    if len(history_vecs) == 0:
        continue
        
    #截断与补零
    if len(history_vecs) >= MAX_SEQ_LEN:
        history_vecs = history_vecs[-MAX_SEQ_LEN:]
    else:
        pad_len = MAX_SEQ_LEN - len(history_vecs)
        pad_vecs = [np.zeros(EMBED_DIM) for _ in range(pad_len)]
        history_vecs = pad_vecs + history_vecs
        
    history_matrix = np.array(history_vecs) #单个样本的小矩阵保留
    
    #拆解曝光展现
    imprs = impressions_str.split()
    for impr in imprs:
        target_news_id, label = impr.split('-')
        label = int(label)
        
        if target_news_id in news_embedding_dict:
            target_vector = news_embedding_dict[target_news_id]
            
            X_dev_target_list.append(target_vector)
            X_dev_history_list.append(history_matrix)
            y_dev_list.append(label)

print(f"验证集特征工程完工(采用流式 List 存储，已免疫 MemoryError)")
print(f"验证集候选新闻总样本数: {len(X_dev_target_list)}")
print(f"验证集历史序列总样本数: {len(X_dev_history_list)}")

#封装并增大 Batch Size (直接复用上面 Cell 已经定义好的 RecommendationDataset)
dev_dataset = RecommendationDataset(X_dev_target_list, X_dev_history_list, y_dev_list)
dev_loader = DataLoader(dev_dataset, batch_size=512, shuffle=False) 
print("="*40)


#开启纯粹的前向预测（不计算梯度，不更新参数）
din_model.eval() # 切换为评估模式

dev_preds = []
dev_labels = []
dev_total_loss = 0

with torch.no_grad():
    for batch_target, batch_history, batch_labels in dev_loader:
        batch_target = batch_target.to(device)
        batch_history = batch_history.to(device)
        batch_labels = batch_labels.to(device)
        
        #仅做前向推理
        predictions = din_model(batch_target, batch_history)
        
        #计算验证集交叉熵
        loss = criterion(predictions, batch_labels)
        dev_total_loss += loss.item() * batch_target.size(0)
        
        dev_preds.extend(predictions.cpu().numpy())
        dev_labels.extend(batch_labels.cpu().numpy())

final_dev_loss = dev_total_loss / len(dev_dataset)
final_dev_auc = roc_auc_score(dev_labels, dev_preds)

print(f"验证集（Dev）终极泛化 AUC  评分 : {final_dev_auc:.4f}")
print(f"验证集（Dev）终极交叉熵 LogLoss : {final_dev_loss:.4f}")

#绘制初步的 ROC 曲线图并保存
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

#计算 ROC 曲线
fpr, tpr, thresholds = roc_curve(dev_labels, dev_preds)
roc_auc = auc(fpr, tpr)

#画图
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'DIN (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (假正例率)')
plt.ylabel('True Positive Rate (真正例率)')
plt.title('DIN 模型 ROC 曲线')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.savefig('din_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"ROC 曲线已成功保存为 din_roc_curve.png")

In [ ]:
seq_lengths = [5, 10, 20]
gbdt_lr_auc = [0.6557, 0.6557, 0.6557]
din_auc = [0.6683, 0.5930, 0.6736]  # 完美还原你的真实跑分
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']  # 完美支持中文字体
plt.rcParams['axes.unicode_minus'] = False
plt.figure(figsize=(7.5, 4.5), dpi=150)
plt.plot(seq_lengths, din_auc, marker='o', color='#ff7f0e', linewidth=2.5, markersize=8, label='DIN 模型')
for x, y in zip(seq_lengths, din_auc):
    plt.text(x, y + 0.005, f'{y:.4f}', ha='center', va='bottom', fontsize=10, weight='bold', color='#d62728')

plt.title('超参数敏感性分析：历史行为序列长度对 DIN 模型的影响', fontsize=11, pad=15, weight='bold')
plt.xlabel('最大序列长度 (MAX_SEQ_LEN)', fontsize=10)
plt.ylabel('测试集泛化 AUC', fontsize=10)
plt.xticks(seq_lengths)
plt.ylim(0.55, 0.72)
plt.tight_layout()
plt.savefig('hyperparameter_sensitivity_din.png', bbox_inches='tight')
plt.show()
print("超参数敏感性分析图（3点连线）已成功保存")
plt.figure(figsize=(8.5, 4.5), dpi=150)
x = np.arange(len(seq_lengths))
width = 0.3
rects1 = plt.bar(x - width/2, gbdt_lr_auc, width, label='对照组：GBDT+LR (传统机器学习)', color='#1f77b4', alpha=0.85)
rects2 = plt.bar(x + width/2, din_auc, width, label='实验组：DIN (深度注意力网络)', color='#ff7f0e', alpha=0.9)
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        plt.annotate(f'{height:.4f}',
                     xy=(rect.get_x() + rect.get_width() / 2, height),
                     xytext=(0, 3),  
                     textcoords="offset points",
                     ha='center', va='bottom', fontsize=9, weight='bold')

autolabel(rects1)
autolabel(rects2)

plt.title('全链路精排模型性能横向对比大盘 (Model Comparison)', fontsize=11, pad=15, weight='bold')
plt.xlabel('历史行为序列长度设定', fontsize=10)
plt.ylabel('测试集泛化 AUC', fontsize=10)
plt.xticks(x, [f'序列长度 = {length}' for length in seq_lengths])
plt.ylim(0.50, 0.75)
plt.legend(loc='lower right', frameon=True, fontsize=9)
plt.tight_layout()
plt.savefig('model_comparison_chart.png', bbox_inches='tight')
plt.show()

print("全链路模型 3 维度对比大图已成功保存")